In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/saas_customers_cleaned.csv",
                  parse_dates=["Subscription_Date", "Renewal_Date", "Cancellation_Date"])
df["Revenue"] = df["Monthly_Fee"] * (1 - df["Discount_Percent"] / 100)
print(df.shape)

(35000, 23)


In [2]:
active_customers = df[df["Customer_Status"] == "Active"]

MRR = active_customers["Revenue"].sum()
print(f"MRR (Monthly Recurring Revenue): ${MRR:,.2f}")

MRR (Monthly Recurring Revenue): $1,990,023.59


In [3]:
ARR = MRR * 12
print(f"ARR (Annual Recurring Revenue): ${ARR:,.2f}")

ARR (Annual Recurring Revenue): $23,880,283.07


In [4]:
ARPU = MRR / len(active_customers)
print(f"ARPU (Average Revenue Per User): ${ARPU:,.2f}")

ARPU (Average Revenue Per User): $77.93


In [5]:
arpu_by_plan = active_customers.groupby("Subscription_Plan")["Revenue"].mean().round(2)
print("\nARPU by plan:")
print(arpu_by_plan)


ARPU by plan:
Subscription_Plan
Basic          14.46
Enterprise    239.97
Premium        95.41
Standard       43.29
Name: Revenue, dtype: float64


In [6]:
total_customers = len(df)
churned_customers = len(df[df["Customer_Status"] == "Churned"])

churn_rate = (churned_customers / total_customers) * 100
retention_rate = 100 - churn_rate

print(f"Total customers: {total_customers:,}")
print(f"Churned customers: {churned_customers:,}")
print(f"Churn Rate: {churn_rate:.2f}%")
print(f"Retention Rate: {retention_rate:.2f}%")

Total customers: 35,000
Churned customers: 9,465
Churn Rate: 27.04%
Retention Rate: 72.96%


In [7]:
avg_cltv = df["Lifetime_Value"].mean()
print(f"Average Customer Lifetime Value: ${avg_cltv:,.2f}")

cltv_by_plan = df.groupby("Subscription_Plan")["Lifetime_Value"].mean().round(2).sort_values(ascending=False)
print("\nCLTV by plan:")
print(cltv_by_plan)

Average Customer Lifetime Value: $1,467.62

CLTV by plan:
Subscription_Plan
Enterprise    5576.27
Premium       2173.40
Standard       928.78
Basic          269.09
Name: Lifetime_Value, dtype: float64


In [8]:
# Convert our cumulative churn rate to an approximate monthly rate for this formula to make sense
monthly_churn_rate = churn_rate / (df["Subscription_Date"].max() - df["Subscription_Date"].min()).days * 30.44 / 100

expected_lifetime_months = 1 / (monthly_churn_rate)
cltv_formula = ARPU * expected_lifetime_months

print(f"Approx. monthly churn rate: {monthly_churn_rate*100:.3f}%")
print(f"Expected customer lifetime: {expected_lifetime_months:.1f} months")
print(f"CLTV (formula-based): ${cltv_formula:,.2f}")

Approx. monthly churn rate: 0.564%
Expected customer lifetime: 177.2 months
CLTV (formula-based): $13,812.76


In [9]:
churned_df = df[df["Customer_Status"] == "Churned"].copy()
churned_df["Subscription_Length_Months"] = (
    (churned_df["Cancellation_Date"] - churned_df["Subscription_Date"]).dt.days / 30.44
)

avg_length_churned = churned_df["Subscription_Length_Months"].mean()
print(f"Average subscription length (churned customers): {avg_length_churned:.1f} months")

active_df = df[df["Customer_Status"] == "Active"].copy()
active_df["Tenure_Months"] = (
    (pd.Timestamp("2026-01-01") - active_df["Subscription_Date"]).dt.days / 30.44
)
avg_tenure_active = active_df["Tenure_Months"].mean()
print(f"Average tenure so far (active customers): {avg_tenure_active:.1f} months")

Average subscription length (churned customers): 11.3 months
Average tenure so far (active customers): 24.4 months


In [10]:
upgrade_rate = (df["Upgrade_Count"] > 0).mean() * 100
downgrade_rate = (df["Downgrade_Count"] > 0).mean() * 100

avg_upgrades = df["Upgrade_Count"].mean()
avg_downgrades = df["Downgrade_Count"].mean()

print(f"% of customers who upgraded at least once: {upgrade_rate:.2f}%")
print(f"% of customers who downgraded at least once: {downgrade_rate:.2f}%")
print(f"Average upgrades per customer: {avg_upgrades:.2f}")
print(f"Average downgrades per customer: {avg_downgrades:.2f}")

% of customers who upgraded at least once: 32.54%
% of customers who downgraded at least once: 14.02%
Average upgrades per customer: 0.40
Average downgrades per customer: 0.15


In [11]:
monthly_active = df[df["Customer_Status"] == "Active"].groupby(
    df["Subscription_Date"].dt.to_period("M")
).size()

print("Monthly Active Customers (by signup cohort month) - last 6 months of data:")
print(monthly_active.tail(6))

# Revenue growth: compare first half vs second half of the dataset's timeline
monthly_revenue = df.groupby(df["Subscription_Date"].dt.to_period("M"))["Revenue"].sum()
first_half_avg = monthly_revenue.iloc[:len(monthly_revenue)//2].mean()
second_half_avg = monthly_revenue.iloc[len(monthly_revenue)//2:].mean()
revenue_growth_pct = ((second_half_avg - first_half_avg) / first_half_avg) * 100

print(f"\nAvg monthly revenue (first half of period): ${first_half_avg:,.2f}")
print(f"Avg monthly revenue (second half of period): ${second_half_avg:,.2f}")
print(f"Revenue growth (first half vs second half): {revenue_growth_pct:.2f}%")

Monthly Active Customers (by signup cohort month) - last 6 months of data:
Subscription_Date
2025-07    518
2025-08    529
2025-09    514
2025-10    499
2025-11    496
2025-12    477
Freq: M, dtype: int64

Avg monthly revenue (first half of period): $47,296.34
Avg monthly revenue (second half of period): $48,437.02
Revenue growth (first half vs second half): 2.41%


In [12]:
kpi_summary = f"""SaaS Business Analytics - KPI Summary
=========================================
MRR (Monthly Recurring Revenue): ${MRR:,.2f}
ARR (Annual Recurring Revenue): ${ARR:,.2f}
ARPU (Average Revenue Per User): ${ARPU:,.2f}
CLTV (actual data): ${avg_cltv:,.2f}
Churn Rate: {churn_rate:.2f}%
Retention Rate: {retention_rate:.2f}%
Avg subscription length (churned): {avg_length_churned:.1f} months
Avg tenure so far (active): {avg_tenure_active:.1f} months
Upgrade rate: {upgrade_rate:.2f}%
Downgrade rate: {downgrade_rate:.2f}%
"""

with open("../reports/kpi_summary.txt", "w") as f:
    f.write(kpi_summary)

print(kpi_summary)

SaaS Business Analytics - KPI Summary
MRR (Monthly Recurring Revenue): $1,990,023.59
ARR (Annual Recurring Revenue): $23,880,283.07
ARPU (Average Revenue Per User): $77.93
CLTV (actual data): $1,467.62
Churn Rate: 27.04%
Retention Rate: 72.96%
Avg subscription length (churned): 11.3 months
Avg tenure so far (active): 24.4 months
Upgrade rate: 32.54%
Downgrade rate: 14.02%

